# 4.1 — Agentic Search: RAG with `ls`, `grep` and `read`

In notebook 2.x we built **classic RAG**: chunk the documents, embed the chunks, store them in a vector index, and retrieve the top-k chunks for every question.

In this notebook we build the approach coding agents like Claude Code, Codex and Cursor use to navigate large codebases: **agentic search**. No embeddings. No vector store. No chunking. We give a local model three tools a human would use in a terminal, and we let it decide what to look at.

| Tool | Terminal equivalent | What the agent uses it for |
|------|--------------------|----------------------------|
| `list_files` | `ls` / `find` / glob | "What documents exist?" |
| `grep` | `grep -n` / `rg` | "Where is this term mentioned?" |
| `read_file` | `sed -n '120,180p'` / `cat -n` | "Show me the passage around line 120." |

```
            ┌──────────────────────────────────────────────┐
            │                                              │
 question ─►│  LLM decides ─► tool call ─► tool result ────┤
            │      ▲                                       │
            │      └────────── appended to messages ◄──────┘
            │
            └─► no more tool calls ─► answer with (file:line) citations
```

**What you will learn**

1. Why agentic search is a strong default for local document Q&A.
2. How to design tools for a *small* model (bounded outputs, line numbers, steering messages).
3. How to write the agent loop by hand with `ollama.chat` (~25 lines).
4. How to read an agent's trace to debug its behavior.

## Why agentic search? (the research in 5 bullets)

- **Anthropic recommends starting with agentic search.** Their Agent SDK guidance describes semantic search as *"usually faster than agentic search, but less accurate, more difficult to maintain, and less transparent"*, and suggests adding it only if you need the speed.
- **Grep often wins on accuracy.** The 2026 study *"Is Grep All You Need?"* found that grep generally beat vector retrieval across several agent harnesses. It also found that the harness (tool design and calling style) mattered as much as the retrieval method.
- **Just-in-time context.** The agent keeps lightweight references (file names, line numbers) and loads only the passages it needs. Anthropic calls this *just-in-time* retrieval in their context-engineering guide.
- **Tools must protect the context window.** Anthropic's tool-writing guide recommends *"pagination, range selection, filtering, and/or truncation with sensible default parameter values"* plus error messages that tell the agent what to try next. Small local models need this even more than frontier models.
- **Fewer, sharper tools.** *"More tools don't always lead to better outcomes."* Three well-described tools beat ten overlapping ones, especially for 8B–30B models.

**Where vector RAG still wins:** paraphrase-heavy questions ("the part about saving memory" when the text says "reduce VRAM footprint"), very large corpora, and latency-sensitive apps. A good agent can compensate by trying synonyms, and you can always add a `semantic_search` tool later (see the exercises).

Sources:
- [Anthropic — Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents)
- [Anthropic — Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
- [Is Grep All You Need? How Agent Harnesses Reshape Agentic Search (arXiv 2605.15184)](https://arxiv.org/abs/2605.15184)
- [Ollama — Tool calling](https://docs.ollama.com/capabilities/tool-calling)

## Setup

You need Ollama running and one tool-calling model pulled:

```bash
ollama pull gemma4        # ~10 GB, runs on 16 GB laptops
ollama pull qwen3.8       # stronger tool use, needs ~24 GB+ of memory
```

In [1]:
%pip install -q ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import re
import ollama

MODEL = "gemma4"          # or "qwen3.8" (stronger at multi-step tool use, slower)
NUM_CTX = 32768           # Ollama's default context is small; agents need room for tool results
THINK = False             # True lets the model reason before each tool call (slower, sometimes smarter)
TEMPERATURE = 0           # agents want repeatable tool calls; 0 made gemma4 noticeably more reliable here

DOCS_DIR = Path("text_files").resolve()   # the corpus the agent is allowed to search
MAX_TOOL_CHARS = 6000                     # hard cap on any single tool result

print("Model:", MODEL)
print("Docs: ", DOCS_DIR)

Model: gemma4
Docs:  /Users/greatmaster/Desktop/projects/oreilly-live-trainings/llama2_oreilly_live_training/notebooks/text_files


Our corpus is four papers extracted from PDF into plain text. Note that the PDF extraction left some two-column text interleaved on the same line. Real documents are messy, and the agent must cope with it.

## Step 1 — Two helpers: a path guard and a line reader

- `_resolve` keeps the agent inside the docs folder. The model chooses the paths it reads, and a tool that accepts `../../.ssh/id_rsa` is a security hole.
- `_lines` is the single place where files get split into lines. `grep` hands out line numbers and `read_file` consumes them, so both must count lines the same way. (PDF text contains form-feed characters: `str.splitlines()` treats them as line breaks, while `for line in file` does not. Mixing the two shifts line numbers by a few lines and breaks citations.)

In [3]:
def _resolve(path: str) -> Path:
    # Turn a relative path into an absolute one, and refuse anything outside DOCS_DIR.
    p = (DOCS_DIR / path).resolve()
    if not p.is_relative_to(DOCS_DIR):
        raise ValueError(f"'{path}' is outside the docs folder. Use paths relative to it, e.g. 'lora-paper.txt'.")
    return p


def _lines(p: Path) -> list[str]:
    # ONE way to split files into lines, shared by every tool.
    # If grep and read_file split lines differently, line numbers stop matching and citations break.
    return p.read_text(errors="ignore").splitlines()

## Step 2 — Tool 1: `list_files` (the agent's `ls`)

Design choices:
- Return **sizes in lines**, so the agent knows a file is too big to read in one go.
- Accept a **glob pattern**, so the agent can narrow to `*.md` or `reports/**` in bigger corpora.

> **Ollama tip:** the Python client turns a plain function into a tool schema automatically. It reads the **type hints** and the **Google-style docstring** (`Args:` section). The docstring *is* the prompt the model sees, so write it for the model.

In [4]:
def list_files(pattern: str = "**/*") -> str:
    """List the documents available to search, with their size in lines.

    Args:
        pattern: Glob pattern relative to the docs folder, e.g. "**/*" (everything) or "*.txt".

    Returns:
        One line per file: "<path> (<n> lines)".
    """
    files = sorted(p for p in DOCS_DIR.glob(pattern) if p.is_file())
    if not files:
        return f"No files match '{pattern}'. Try pattern='**/*'."
    lines = []
    for p in files:
        lines.append(f"{p.relative_to(DOCS_DIR)} ({len(_lines(p))} lines)")
    return "\n".join(lines)

print(list_files())

instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)


## Step 3 — Tool 2: `grep` (the agent's search engine)

This is the most important tool. It replaces the vector index.

Design choices, each taken from the best practices above:
- **Regex + case-insensitive by default.** The model can search `NF4|NormalFloat` in one call.
- **Every hit carries `file:line`.** The line number is the "pointer" the agent passes to `read_file` next.
- **A per-file count summary first.** The agent sees which document is *about* the topic before it reads anything.
- **Bounded output** (`max_matches`) with a message that says how many hits were hidden and how to narrow the search.
- **Steering messages** on zero hits and on bad regex, instead of a bare error.

We use Python's `re` so the notebook runs on any OS. In a real system you would shell out to [ripgrep](https://github.com/BurntSushi/ripgrep) for speed.

In [5]:
def grep(pattern: str, glob: str = "**/*", max_matches: int = 20) -> str:
    """Search all documents for a regular expression (case-insensitive). Returns matching lines with file:line.

    Use this to find WHERE a topic is discussed. Then call read_file around the line numbers to see full context.

    Args:
        pattern: Regular expression, e.g. "NormalFloat|NF4" or "learning rate". Keep it short and distinctive.
        glob: Which files to search, e.g. "**/*" (all) or "qlora*".
        max_matches: Maximum matching lines to return (default 20).

    Returns:
        A per-file hit count, then lines formatted as "file:line: text".
    """
    try:
        rx = re.compile(pattern, re.IGNORECASE)
    except re.error as e:
        return f"Invalid regex ({e}). Escape special characters like ( ) [ ] . with a backslash, or search a plain word."

    hits, counts = [], {}
    for p in sorted(DOCS_DIR.glob(glob)):
        if not p.is_file():
            continue
        rel = str(p.relative_to(DOCS_DIR))
        for i, line in enumerate(_lines(p), start=1):
            if rx.search(line):
                counts[rel] = counts.get(rel, 0) + 1
                hits.append(f"{rel}:{i}: {line.strip()[:200]}")

    if not hits:
        return f"No matches for '{pattern}'. Try a shorter keyword, a synonym, or an abbreviation."

    summary = "Matches per file: " + ", ".join(f"{f}={n}" for f, n in counts.items())
    shown = hits[:max_matches]
    out = summary + "\n\n" + "\n".join(shown)
    if len(hits) > max_matches:
        out += f"\n\n[{len(hits) - max_matches} more matches hidden. Narrow the pattern or set glob to one file.]"
    return out

print(grep("NormalFloat|NF4", max_matches=5))

Matches per file: qlora-paper.txt=27

qlora-paper.txt:26: without sacrificing performance: (a) 4-bit NormalFloat (NF4), a new data type that
qlora-paper.txt:84: formance: (1) 4-bit NormalFloat, an information theoretically optimal quantization data type for
qlora-paper.txt:182: QL O RA achieves high-fidelity 4-bit finetuning via two techniques we propose—4-bit NormalFloat
qlora-paper.txt:183: (NF4) quantization and Double Quantization. Additionally, we introduce Paged Optimizers, to
qlora-paper.txt:190: 4-bit NormalFloat Quantization The NormalFloat (NF) data type builds on Quantile Quantization

[22 more matches hidden. Narrow the pattern or set glob to one file.]


Look at the output. The agent learns three things from one call: which file covers the topic, how often, and the exact lines to open next.

Now see the steering messages the model gets when it makes a mistake:

In [8]:
print(grep("photosynthesis"))
print()
print(grep("LoRAConfig"))

No matches for 'photosynthesis'. Try a shorter keyword, a synonym, or an abbreviation.

Matches per file: instruction-tune-llama2-extended-guide.txt=3

instruction-tune-llama2-extended-guide.txt:346: to create our LoRAConfig and provide it to the trainer.
instruction-tune-llama2-extended-guide.txt:349: from peft import LoraConfig, prepare_model_for_kbit_training, get_pef
instruction-tune-llama2-extended-guide.txt:358: peft_config = LoraConfig(


## Step 4 — Tool 3: `read_file` (read a *window*, never the whole file)

`sparks-agi-paper.txt` has 9,000+ lines. Dumping it into an 8B model's context would push out the question itself. So `read_file` reads a **window** of lines:

- `offset` + `limit` work like `sed -n '120,180p'`.
- Output uses **`cat -n` style line numbers**, so the agent can cite `qlora-paper.txt:187`.
- A **footer** tells the agent where it is and how to continue (pagination for LLMs).

In [9]:
def read_file(path: str, offset: int = 1, limit: int = 60) -> str:
    """Read a window of lines from a document, with line numbers.

    Use after grep: set offset a few lines before a match to read its surrounding context.

    Args:
        path: File path relative to the docs folder, e.g. "qlora-paper.txt".
        offset: First line to read (1-based).
        limit: Number of lines to read (default 60, max 200).

    Returns:
        Numbered lines, followed by a footer saying which lines were shown.
    """
    p = _resolve(path)
    if not p.is_file():
        return f"File '{path}' not found. Call list_files to see valid paths."
    lines = _lines(p)
    start = max(offset, 1)
    end = min(start + min(limit, 200) - 1, len(lines))
    body = "\n".join(f"{n:>5}\t{lines[n - 1]}" for n in range(start, end + 1))
    footer = f"[{path}: lines {start}-{end} of {len(lines)}"
    footer += f". Call read_file with offset={end + 1} to continue.]" if end < len(lines) else ". End of file.]"
    return body + "\n" + footer

print(read_file("qlora-paper.txt", offset=185, limit=10))

  185	traditionally made finetuning on a single machine difficult for large models.
  186	QL O RA has one low-precision storage data type, in our case usually 4-bit, and one computation data
  187	type that is usually BFloat16. In practice, this means whenever a QL O RA weight tensor is used, we
  188	dequantize the tensor to BFloat16, and then perform a matrix multiplication in 16-bit.
  189	We now discuss the components of QL O RA followed by a formal definition of QL O RA.
  190	4-bit NormalFloat Quantization The NormalFloat (NF) data type builds on Quantile Quantization
  191	[15] which is an information-theoretically optimal data type that ensures each quantization bin has an
  192	equal number of values assigned from the input tensor. Quantile quantization works by estimating
  193	the quantile of the input tensor through the empirical cumulative distribution function.
  194	The main limitation of quantile quantization is that the process of quantile estimation is expensive.
[qlo

### Quick check: do the tools behave?

Agents fail silently when a tool is broken. They just "reason around" the bad output. Test tools on their own before you hand them to a model.

In [10]:
assert "lora-paper.txt" in list_files()
assert "qlora-paper.txt:" in grep("NormalFloat")
assert "No matches" in grep("zzzz-not-in-corpus")
assert "Invalid regex" in grep("(")
# the line number grep reports must point at the same text in read_file
hit = grep("4-bit NormalFloat Quantization").splitlines()[2]      # "qlora-paper.txt:<n>: ..."
_, n, text = hit.split(":", 2)
assert text.strip() in read_file("qlora-paper.txt", offset=int(n), limit=1)
try:
    read_file("../../README.md")
    raise AssertionError("path escape was allowed!")
except ValueError:
    pass
print("All tool checks passed.")

All tool checks passed.


## Step 5 — The system prompt: teach the *search workflow*

Small models do much better when you spell out the procedure an expert would follow. This prompt encodes the loop a human researcher runs in a terminal: **list → grep → read → answer with citations**.

Two rules matter most:
- **"Use only the documents."** That rule grounds the answer in the corpus.
- **"Say so if the answer isn't there."** That rule gives the model permission to not know, which reduces hallucination.

In [11]:
SYSTEM_PROMPT = '''You are a research assistant. You answer questions using ONLY the documents in a local folder.
You cannot see the documents until you use your tools.

Workflow:
1. Call list_files once to see which documents exist.
2. Call grep with short, distinctive keywords from the question. If nothing matches, retry with synonyms or abbreviations (2-3 attempts).
3. Call read_file around the most promising line numbers. grep shows single lines; read_file shows the full passage.
4. Repeat 2-3 if you still miss a piece of the answer.
5. Answer concisely. Cite every claim as (file:line).
   If the documents do not contain the answer, say "The documents don't say." Do not use outside knowledge.'''

## Step 6 — The agent loop

This is the whole "framework". Each step:

1. Send the full message history plus the tool list to the model.
2. If the model replied with **no tool calls**, its text is the final answer. Stop.
3. Otherwise run each requested tool and append the result as a `role: "tool"` message.
4. Go back to 1.

`max_steps` is a circuit breaker. A confused small model can loop forever on `grep`.

In [12]:
TOOLS = [list_files, grep, read_file]
TOOLS_BY_NAME = {f.__name__: f for f in TOOLS}


def run_tool(name: str, args: dict) -> str:
    fn = TOOLS_BY_NAME.get(name)
    if fn is None:
        return f"Unknown tool '{name}'. Available tools: {list(TOOLS_BY_NAME)}."
    try:
        result = fn(**args)
    except Exception as e:  # return errors to the model so it can fix its call
        result = f"Error calling {name}: {e}"
    if len(result) > MAX_TOOL_CHARS:
        result = result[:MAX_TOOL_CHARS] + "\n[Output truncated. Request a smaller window or a narrower search.]"
    return result


def run_agent(question: str, model: str = MODEL, max_steps: int = 12, verbose: bool = True):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = ollama.chat(
            model=model,
            messages=messages,
            tools=TOOLS,
            think=THINK,
            options={"num_ctx": NUM_CTX, "temperature": TEMPERATURE},
        )
        msg = response.message
        messages.append(msg)

        if not msg.tool_calls:  # no tool call = final answer
            return msg.content, messages

        for call in msg.tool_calls:
            name, args = call.function.name, call.function.arguments
            result = run_tool(name, args)
            if verbose:
                preview = result if len(result) < 400 else result[:400] + " ..."
                print(f"── step {step}: {name}({args})\n{preview}\n")
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return f"Stopped after {max_steps} steps without a final answer.", messages

## Step 7 — Run it

### Question 1: a single fact

Watch the trace. You should see the pattern `list_files → grep → read_file → answer`.

In [13]:
answer, trace = run_agent("How much GPU memory does QLoRA need to finetune a 65B parameter model?")
print("=" * 70)
print(answer)

── step 1: list_files({'pattern': '**/*'})
instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)

── step 2: grep({'glob': '**/*', 'max_matches': 5, 'pattern': '65B'})
Matches per file: qlora-paper.txt=46

qlora-paper.txt:19: age enough to finetune a 65B parameter model on a single 48GB GPU while
qlora-paper.txt:33: finetuning (e.g. 33B and 65B parameter models). Our results show that QLoRA
qlora-paper.txt:46: finetuning very large models is prohibitively expensive; regular 16-bit finetuning of a LLaMA 65B
qlora-paper.txt:66: of finetuning a 65B parameter model  ...

── step 3: read_file({'limit': 10, 'offset': 19, 'path': 'qlora-paper.txt'})
   19	                                                     age enough to finetune a 65B parameter model on a single 48GB GPU while
   20	                                                     preserving full 16-bit finetuning task performance. QL O RA backpro

### Baseline: the same model *without* tools

For contrast, ask the bare model. The paper's answer is **a single 48 GB GPU**. Without tools, the model answers from memory: it may be right or confidently wrong, and it cannot cite a source. With tools, every claim points to a line you can open.

In [15]:
closed_book = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "How much GPU memory does QLoRA need to finetune a 65B parameter model?"}],
    think=THINK,
)
print(closed_book.message.content)

Finishingly, the GPU memory required to QLoRA fine-tune a 65B parameter model is **extremely high** and will likely require **multiple, top-tier, high-memory GPUs (like A100 80GB or H100 80GB)** connected via high-speed interconnects (like NVLink) in a specialized cluster setup.

Here is a detailed breakdown of the factors involved, the estimation, and the realistic limitations.

---

## 🧠 The Theoretical Calculation (The "Why It's Hard")

While QLoRA significantly reduces memory usage compared to full fine-tuning, the size of the base model ($N$) is the dominant factor:

**Base Model Size:** 65 Billion parameters ($\text{65B}$)

**QLoRA Memory Savings:** QLoRA quantizes the base weights to 4-bit precision (saving $\approx 75\%$ of the memory compared to full FP16 storage).

### 1. Storing the Base Model Weights (The Biggest Cost)

Even with 4-bit quantization, you still need to load the base weights into memory.

* **Quantization Math:** 1 parameter $\times$ 4 bits/parameter $\approx 

### Question 2: a concept that needs reading, not just matching

`grep` finds the term. The agent must then **read** the surrounding passage to explain it.

In [16]:
answer, trace = run_agent("What is the NF4 data type in QLoRA and why is it a good choice for quantizing model weights?")
print("=" * 70)
print(answer)

── step 1: list_files({'pattern': '**/*'})
instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)

── step 2: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'NF4'})
Matches per file: qlora-paper.txt=19

qlora-paper.txt:26: without sacrificing performance: (a) 4-bit NormalFloat (NF4), a new data type that
qlora-paper.txt:183: (NF4) quantization and Double Quantization. Additionally, we introduce Paged Optimizers, to
qlora-paper.txt:262: NF4
qlora-paper.txt:278: We use NF4 for W and FP8 for c2 . We use a blocksize of 64 for W for higher quantization precision
q ...

── step 3: read_file({'limit': 10, 'offset': 20, 'path': 'qlora-paper.txt'})
   20	                                                     preserving full 16-bit finetuning task performance. QL O RA backpropagates gradi-
   21	                                                     ents through a frozen, 4-bit quantized pretrained languag

### Question 3: multi-hop across two documents

The agent needs evidence from `lora-paper.txt` **and** `qlora-paper.txt`. Count how many `grep` calls it makes. A strong run searches each paper separately. A weak run stays inside `qlora-paper.txt` (which also describes LoRA) and never opens the LoRA paper. Small models often take that shortcut, and the trace shows it.

In [14]:
answer, trace = run_agent(
    "Compare how LoRA and QLoRA each reduce memory usage during finetuning. "
    "What does QLoRA add on top of LoRA?"
)
print("=" * 70)
print(answer)

── step 1: list_files({'pattern': '**/*'})
instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)



── step 2: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'LoRA memory'})
No matches for 'LoRA memory'. Try a shorter keyword, a synonym, or an abbreviation.



── step 3: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'QLoRA memory'})
No matches for 'QLoRA memory'. Try a shorter keyword, a synonym, or an abbreviation.



── step 4: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'LoRA QLoRA memory'})
No matches for 'LoRA QLoRA memory'. Try a shorter keyword, a synonym, or an abbreviation.



── step 5: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'memory reduction'})
No matches for 'memory reduction'. Try a shorter keyword, a synonym, or an abbreviation.



── step 6: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'quantization'})
Matches per file: instruction-tune-llama2-extended-guide.txt=3, qlora-paper.txt=55

instruction-tune-llama2-extended-guide.txt:251: Quantization-aware Low-Rank Adapter Tuning for Language
instruction-tune-llama2-extended-guide.txt:264: bitsandbytes, 4-bit quantization and QLoRA blog post.
instruction-tune-llama2-extended-guide.txt:328: quantization_config=bnb_config,
qlora-paper.txt:28: Quantizati ...



── step 7: read_file({'limit': 10, 'offset': 250, 'path': 'instruction-tune-llama2-extended-guide.txt'})
  250	                                                     We will use the recently introduced method in the paper "QLoRA:
  251	                                                     Quantization-aware Low-Rank Adapter Tuning for Language
  252	                                                     Generation" by Tim Dettmers et al. QLoRA is a new technique to
  253	                                   ...



── step 8: read_file({'limit': 10, 'offset': 28, 'path': 'qlora-paper.txt'})
   28	                                                     Quantization to reduce the average memory footprint by quantizing the quantization
   29	                                                     constants, and (c) Paged Optimizers to manage memory spikes. We use QL O RA
   30	                                                     to finetune more than 1,000 models, providing a detailed analysi ...



── step 9: read_file({'limit': 10, 'offset': 328, 'path': 'instruction-tune-llama2-extended-guide.txt'})
  328	                                                           quantization_config=bnb_config,
  329	                                                           use_cache=False,
  330	                                                           use_flash_attention_2=use_flash_attention,
  331	                                                           device_map="auto",
  332	                        ...



The documents indicate that QLoRA is a technique designed to reduce the memory footprint during finetuning (instruction-tune-llama2-extended-guide.txt:253, 254).

Here is a comparison of how LoRA and QLoRA reduce memory usage:

*   **LoRA:** The process involves attaching small, trainable adapter layers to the model (instruction-tune-llama2-extended-guide.txt:258).
*   **QLoRA:** QLoRA builds upon LoRA by first **quantizing the pre-trained model to 4 bits and freezing it** (instruction-tune-llama2-extended-guide.txt:257). It then attaches the small, trainable adapter layers (LoRA) and finetunes only those layers while using the frozen, quantized model (instruction-tune-llama2-extended-guide.txt:258-259).

**What QLoRA adds on top of LoRA:**
QLoRA adds **4-bit quantization** of the pre-trained model, which is then frozen, to the standard LoRA technique (instruction-tune-llama2-extended-guide.txt:257). This quantization step is key to further reducing the memory footprint (qlora-paper.tx

### Question 4: the answer is NOT in the documents

A good RAG system says "I don't know" when it should. This is the most important test. A vector store *always* returns top-k chunks, even irrelevant ones. Agentic search returns `No matches`, a clearer signal for the model.

Watch for false positives in the trace: the regex `Llama 3` also matches "LLaMA **3**3B". The agent has to read the passage to notice that it is a different model.

In [15]:
answer, trace = run_agent("What is the release date of Llama 3?")
print("=" * 70)
print(answer)

── step 1: list_files({'pattern': '**/*'})
instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)



── step 2: grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'Llama 3'})
Matches per file: qlora-paper.txt=1

qlora-paper.txt:556: a LLaMA 33B model finetuned with Reinforcement Learning from Human Feedback (RLHF) on



── step 3: read_file({'limit': 10, 'offset': 550, 'path': 'qlora-paper.txt'})
  550	optimizers to prevent memory spikes during gradient checkpointing. We do small hyperparameter
  551	searches for the 13B and 33B LLaMA models and we find that all hyperparameter settings found
  552	at 7B generalize (including number of epochs) except learning rate and batch size. We halve the
  553	learning rate for 33B and 65B while doubling the batch size.
  554	Baselines We compare our m ...



The documents don't say.


## Step 8 — Read the trace

The `trace` is the full message list. Reading it is how you debug an agent. Ask:

- Did it **grep before reading**, or did it read files blindly?
- Did it **retry with synonyms** after a miss?
- Did it read **near the matched line numbers**?
- Do the **citations** in the answer match lines it actually read?

In [17]:
def summarize_trace(trace):
    calls = [(c.function.name, c.function.arguments)
             for m in trace if getattr(m, "tool_calls", None)
             for c in m.tool_calls]
    for i, (name, args) in enumerate(calls, 1):
        print(f"{i:>2}. {name}({args})")
    tool_chars = sum(len(m["content"]) for m in trace if isinstance(m, dict) and m.get("role") == "tool")
    print(f"\n{len(calls)} tool calls, {tool_chars:,} characters of tool output sent to the model")

summarize_trace(trace)

 1. list_files({'pattern': '**/*'})
 2. grep({'glob': '**/*', 'max_matches': 5, 'pattern': 'NF4'})
 3. read_file({'limit': 10, 'offset': 20, 'path': 'qlora-paper.txt'})
 4. read_file({'limit': 10, 'offset': 250, 'path': 'qlora-paper.txt'})
 5. read_file({'limit': 10, 'offset': 260, 'path': 'qlora-paper.txt'})
 6. read_file({'limit': 10, 'offset': 270, 'path': 'qlora-paper.txt'})

6 tool calls, 4,596 characters of tool output sent to the model


Compare the last number to the corpus size: the four papers total ~13,000 lines (~700K characters). The agent answered while loading a tiny fraction of that into context. That is *just-in-time retrieval*.

## Step 9 — Try the other model

Tool-calling skill varies a lot between local models. Run the multi-hop question with `qwen3.8` (if you have it) and compare the traces: number of steps, quality of grep patterns, citation accuracy.

In [17]:
# Requires: ollama pull qwen3.8  (picks the first local qwen3.8 tag, e.g. qwen3.8:27b)
QWEN = next((m.model for m in ollama.list().models if m.model.startswith("qwen3.8")), "qwen3.8")
print("Using", QWEN)

answer, trace = run_agent(
    "Compare how LoRA and QLoRA each reduce memory usage during finetuning. "
    "What does QLoRA add on top of LoRA?",
    model=QWEN,
)
print("=" * 70)
print(answer)
summarize_trace(trace)

Using qwen3.8:27b-mlx-16k


── step 1: list_files({'pattern': '**/*'})
instruction-tune-llama2-extended-guide.txt (605 lines)
lora-paper.txt (1803 lines)
qlora-paper.txt (1522 lines)
sparks-agi-paper.txt (9527 lines)



── step 2: grep({'pattern': 'memory', 'glob': 'qlora-paper.txt', 'max_matches': 30})
Matches per file: qlora-paper.txt=51

qlora-paper.txt:18: We present QL O RA, an efficient finetuning approach that reduces memory us-
qlora-paper.txt:25: on a single GPU. QL O RA introduces a number of innovations to save memory
qlora-paper.txt:28: Quantization to reduce the average memory footprint by quantizing the quantization
qlora-paper.txt:29: constants, and (c) Paged Optimizers to manage m ...

── step 2: grep({'pattern': 'memory', 'glob': 'lora-paper.txt', 'max_matches': 30})
Matches per file: lora-paper.txt=5

lora-paper.txt:27: GPU memory requirement by 3 times. LoRA performs on-par or better than fine-
lora-paper.txt:149: compute- and memory-efficient. When the pre-trained model is GPT-3 175B, the number of train-
lora-paper.txt:243: then adding a different B 0 A0 , a quick operation with very little memory overhead. Critically, this
lora-paper.txt:268: memory and s ...



── step 3: read_file({'path': 'qlora-paper.txt', 'offset': 15, 'limit': 40})
   15	
   16	
   17	                                                                                        Abstract
   18	                                                     We present QL O RA, an efficient finetuning approach that reduces memory us-
   19	                                                     age enough to finetune a 65B parameter model on a single 48GB GPU while
   20	           ...

── step 3: read_file({'path': 'qlora-paper.txt', 'offset': 150, 'limit': 110})
  150	constants ci .
  151	Low-rank Adapters Low-rank Adapter (LoRA) finetuning [28] is a method that reduces memory
  152	requirements by using a small set of trainable parameters, often termed adapters, while not updating
  153	the full model parameters which remain fixed. Gradients during stochastic gradient descent are
  154	passed through the fixed pretrained model weights to the adapter, whi ...



── step 4: read_file({'limit': 60, 'offset': 220, 'path': 'qlora-paper.txt'})
  220	a symmetric k-bit quantization is that this approach does not have an exact representation of zero,
  221	which is an important property to quantize padding and other zero-valued elements with no error. To
  222	
  223	
  224	                                                         4
  225	
  226	ensure a discrete zeropoint of 0 and to use all 2k bits for a k-bit datatype, we create an asymm ...

── step 4: read_file({'limit': 40, 'offset': 255, 'path': 'qlora-paper.txt'})
  255	memory when the memory is needed in the optimizer update step.
  256	QL O RA. Using the components described above, we define QL O RA for a single linear layer in
  257	the quantized base model with a single LoRA adapter as follows:
  258	
  259	                   YBF16 = XBF16 doubleDequant(cFP32
  260	                                                1    , ck-bit
  261	                      ...



...

── step 7: read_file({'limit': 30, 'offset': 100, 'path': 'qlora-paper.txt'})
  100	generalization. Second, we show that strong Massive Multitask Language Understanding (MMLU)
  101	benchmark performance does not imply strong Vicuna chatbot benchmark performance and vice
  102	versa—in other words, dataset suitability matters more than size for a given task.
  103	Furthermore, we also provide a extensive analysis of chatbot performance that uses both human
  104	raters and  ...

── step 7: read_file({'limit': 30, 'offset': 120, 'path': 'lora-paper.txt'})
  120	yi are sequences of tokens. For example, in NL2SQL, xi is a natural language query and yi its
  121	corresponding SQL command; for summarization, xi is the content of an article and yi its summary.
  122	
  123	
  124	                                                  2
  125	
  126	During full fine-tuning, the model is initialized to pre-trained weights Φ0 and updated to Φ0 + ∆Φ
  127	by rep ...



── step 8: read_file({'limit': 30, 'offset': 170, 'path': 'lora-paper.txt'})
  170	parallelism to keep the latency low, and adapter layers have to be processed sequentially. This makes
  171	a difference in the online inference setting where the batch size is typically as small as one. In a
  172	generic scenario without model parallelism, such as running inference on GPT-2 (Radford et al., b)
  173	medium on a single GPU, we see a noticeable increase in latency when using ...

── step 8: read_file({'limit': 30, 'offset': 55, 'path': 'qlora-paper.txt'})
   55	                                             2
   56	                                                 https://github.com/artidoro/qlora and https://github.com/TimDettmers/bitsandbytes
   57	
   58	
   59	                                         Preprint. Under review.
   60	
   61	that are tuned by backpropagating gradients through Table 1: Elo ratings for a competition between
   62	        ...



── step 9: read_file({'limit': 15, 'offset': 83, 'path': 'qlora-paper.txt'})
   83	QL O RA introduces multiple innovations designed to reduce memory use without sacrificing per-
   84	formance: (1) 4-bit NormalFloat, an information theoretically optimal quantization data type for
   85	normally distributed data that yields better empirical results than 4-bit Integers and 4-bit Floats.
   86	(2) Double Quantization, a method that quantizes the quantization constants, saving ...

── step 9: read_file({'limit': 30, 'offset': 200, 'path': 'lora-paper.txt'})
  200	describe in Section 5.1. The inference latency introduced by adapter layers can be significant in an
  201	online, short-sequence-length scenario. See the full study in Appendix B.
  202	
  203	
  204	4       O UR M ETHOD
  205	We describe the simple design of LoRA and its practical benefits. The principles outlined here apply
  206	to any dense layers in deep learning models, though we only ...



── step 10: read_file({'limit': 15, 'offset': 25, 'path': 'lora-paper.txt'})
   25	                                                   rameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam,
   26	                                                   LoRA can reduce the number of trainable parameters by 10,000 times and the
   27	                                                   GPU memory requirement by 3 times. LoRA performs on-par or better than fine-
  ...



Now I have all the information needed to provide a thorough comparison.

---

## How LoRA and QLoRA Reduce Memory During Finetuning

### LoRA's Memory Reduction

LoRA reduces memory by **freezing the pretrained model weights** and only training a small set of low-rank adapter matrices. Specifically:

- For a pretrained weight matrix W₀, LoRA constrains the update to a low-rank decomposition: W₀ + ΔW = W₀ + BA, where B ∈ ℝ^(d×r) and A ∈ ℝ^(r×k), with rank r ≪ min(d, k) (lora-paper.txt:216–218).
- During training, W₀ is frozen and does not receive gradient updates; only A and B are trainable (lora-paper.txt:219).
- This reduces the number of trainable parameters by up to **10,000×** and GPU memory by **3×** compared to full fine-tuning with Adam (lora-paper.txt:25–27).
- The key savings come from not storing optimizer states for the frozen parameters. For GPT-3 175B, VRAM drops from 1.2 TB to 350 GB, and checkpoint size from 350 GB to 35 MB (lora-paper.txt:268–273).
- LoRA also avoids ad

## Recap: best practices for local agentic-search tools

| Practice | Where we applied it |
|----------|--------------------|
| Few, sharp tools | 3 tools mirroring `ls`, `grep`, `sed -n` |
| Descriptions written for the model | Google-style docstrings → Ollama tool schema |
| Pointers, not payloads | `grep` returns `file:line`; `read_file` loads a window |
| Bounded outputs with defaults | `max_matches=20`, `limit=60`, `MAX_TOOL_CHARS` |
| Pagination hints | `read_file` footer: "Call read_file with offset=…" |
| Errors that steer | zero hits → "try a synonym"; bad regex → "escape it" |
| Sandboxed paths | `_resolve()` blocks `../` escapes |
| Explicit workflow in the prompt | list → grep → read → cite |
| Permission to not know | "The documents don't say." |
| Circuit breaker | `max_steps` |
| Enough context | `num_ctx=32768` (Ollama's default is far smaller) |
| Repeatable tool calls | `temperature=0` |

## Exercises

1. **Point it at your own notes.** Change `DOCS_DIR` to a folder of `.md` files (an Obsidian vault works well) and ask questions about it.
2. **Break it on purpose.** Set `max_matches=200` and `limit=1000`. Run Question 3 again. What happens to answer quality on `gemma4`?
3. **Paraphrase stress test.** Ask "How do the authors make the optimizer state cheaper?" The text says "paged optimizers". Does the agent find it by trying synonyms?
4. **Hybrid search.** Add a fourth tool, `semantic_search(query)`, built on the `nomic-embed-text` embeddings from notebook 2.1. Does the agent learn when to use which tool?
5. **Turn on thinking.** Set `THINK = True` and compare the number of tool calls and the total time.